# 🏆 Projeto Final — Análise Exploratória do IDHM nos Municípios Brasileiros

**Curso Introdutório de Python para Ciência de Dados**  
Universidade de Fortaleza — UNIFOR | Disciplina T326

---

## 📋 Ficha do Projeto

| Campo | Detalhe |
|-------|--------|
| **Dataset** | Brazilian Cities Dataset — 5.570 municípios brasileiros |
| **Fonte** | IBGE / Atlas Brasil |
| **Variável-Alvo** | IDHM (Índice de Desenvolvimento Humano Municipal) |
| **Tipo de Análise** | Exploratória (EDA) + Storytelling com Dados |
| **Ferramentas** | Python, Pandas, Matplotlib, Seaborn, NumPy |

---

## ❓ Pergunta de Negócio

> **"O que determina o nível de desenvolvimento humano (IDHM) de um município brasileiro, e como ele varia entre regiões, estados e categorias urbanas?"**

Esta é uma pergunta com alto impacto para gestores públicos, pesquisadores e a sociedade civil: entender os fatores associados ao IDHM pode guiar políticas públicas mais efetivas.

---

## 🧭 Perguntas Norteadoras

| # | Pergunta |
|---|----------|
| 1 | Como o IDHM varia entre as cinco regiões do Brasil? |
| 2 | Quais variáveis têm maior correlação com o IDHM? |
| 3 | Qual o grau de desigualdade interna nos estados? |
| 4 | Cidades mais urbanizadas têm sistematicamente IDHM mais alto? |
| 5 | Qual componente do IDHM é o "gargalo" em cada região? |
| 6 | Municípios maiores são necessariamente mais desenvolvidos? |

---

## 🗂️ Estrutura do Notebook

```
0. Configuração
1. Carregamento e Limpeza de Dados
2. Análise Univariada
3. Análise Geográfica (Pergunta 1)
4. Análise de Correlação (Pergunta 2)
5. Desigualdade Interna (Pergunta 3)
6. Urbanização e Porte (Perguntas 4 e 6)
7. Componentes do IDHM (Pergunta 5)
8. Análise Multivariada
9. Painel Executivo (Dashboard)
10. Conclusões e Insights
```

---
## ⚙️ 0. Configuração do Ambiente

In [ ]:
# =============================================================
# PROJETO FINAL — Análise do IDHM nos Municípios Brasileiros
# Curso Python para Ciência de Dados — UNIFOR T326
# =============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# ── Tema visual global ──────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 120,
    'figure.facecolor': 'white',
    'axes.facecolor': '#fafafa',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'axes.grid.axis': 'y',
    'grid.alpha': 0.4,
    'grid.linestyle': '--',
    'font.size': 10,
})

# Paleta de cores por região — consistente em todo o projeto
COR_REGIAO = {
    'Norte':        '#1f77b4',
    'Nordeste':     '#ff7f0e',
    'Centro-Oeste': '#2ca02c',
    'Sudeste':      '#d62728',
    'Sul':          '#9467bd'
}

# Faixas de IDHM (classificação ONU/PNUD)
BINS_IDHM   = [0.0, 0.499, 0.599, 0.699, 0.799, 1.0]
LABELS_IDHM = ['Muito Baixo', 'Baixo', 'Médio', 'Alto', 'Muito Alto']
CORES_FAIXA = ['#d73027', '#fc8d59', '#fee090', '#91cf60', '#1a9641']

print('✅ Configuração concluída.')
print(f'   Pandas  {pd.__version__} | NumPy {np.__version__} | Matplotlib {plt.matplotlib.__version__}')

---
## 📥 1. Carregamento e Preparação dos Dados

In [ ]:
# ── Carregamento ─────────────────────────────────────────────────
try:
    df = pd.read_csv('../dataset/brazil_cities.csv', encoding='latin-1', sep=';')
    print(f'✅ Dataset real carregado: {df.shape}')

except FileNotFoundError:
    print('⚠️  Dataset não encontrado — gerando dados sintéticos realistas para demonstração...')
    np.random.seed(2024)
    regioes_cfg = {
        'Norte':        {'est': ['AM','PA','AC','RO','RR','AP','TO'], 'n': 621,  'mu': 0.614, 'sg': 0.052},
        'Nordeste':     {'est': ['MA','PI','CE','RN','PB','PE','AL','SE','BA'], 'n': 1793, 'mu': 0.599, 'sg': 0.056},
        'Centro-Oeste': {'est': ['MT','MS','GO','DF'], 'n': 467,  'mu': 0.704, 'sg': 0.044},
        'Sudeste':      {'est': ['MG','ES','RJ','SP'], 'n': 1668, 'mu': 0.737, 'sg': 0.050},
        'Sul':          {'est': ['PR','SC','RS'],      'n': 1191, 'mu': 0.749, 'sg': 0.037},
    }
    rows = []
    for reg, cfg in regioes_cfg.items():
        for i in range(cfg['n']):
            idhm = float(np.clip(np.random.normal(cfg['mu'], cfg['sg']), 0.35, 0.950))
            pib  = max(2800, idhm * 57000 + np.random.normal(0, 9500))
            pop  = int(np.clip(np.random.lognormal(9.0, 1.5), 800, 12_000_000))
            area = float(round(np.random.lognormal(7.0, 1.2), 1))
            rows.append({
                'CITY':             f'Municipio_{len(rows):05d}',
                'STATE':            np.random.choice(cfg['est']),
                'REGION':           reg,
                'IDHM':             round(idhm, 3),
                'IDHM_Renda':       round(float(np.clip(idhm + np.random.normal(0.00, 0.031), 0.30, 0.95)), 3),
                'IDHM_Longevidade': round(float(np.clip(idhm + np.random.normal(0.025, 0.020), 0.40, 0.95)), 3),
                'IDHM_Educacao':    round(float(np.clip(idhm - np.random.normal(0.026, 0.032), 0.25, 0.90)), 3),
                'GDP_CAPITA':       round(pib, 2),
                'IBGE_POP':         pop,
                'AREA':             area,
            })
    df = pd.DataFrame(rows)
    df['DENSIDADE'] = (df['IBGE_POP'] / df['AREA']).round(2)
    df['GDP']       = (df['GDP_CAPITA'] * df['IBGE_POP']).round(0).astype(int)
    print(f'✅ Dataset sintético: {df.shape[0]} municípios, {df.shape[1]} colunas')

In [ ]:
# ── Engenharia de Features ────────────────────────────────────────
# Criamos colunas derivadas úteis para as análises seguintes

# 1. Faixa de IDHM
df['FAIXA_IDHM'] = pd.cut(df['IDHM'], bins=BINS_IDHM, labels=LABELS_IDHM, include_lowest=True)

# 2. Porte do município por população
df['PORTE'] = pd.cut(
    df['IBGE_POP'],
    bins=[0, 5_000, 20_000, 100_000, 500_000, float('inf')],
    labels=['Muito Pequeno\n(<5k)', 'Pequeno\n(5–20k)',
            'Médio\n(20–100k)', 'Grande\n(100–500k)', 'Metrópole\n(>500k)']
)

# 3. Log do PIB per capita (reduz skewness para correlações)
df['LOG_GDP_CAPITA'] = np.log1p(df['GDP_CAPITA'])

# 4. Verificação de nulos após engenharia
nulos = df.isnull().sum()
print('=== DADOS APÓS PREPARAÇÃO ===')
print(f'  Forma       : {df.shape}')
print(f'  Valores nulos: {nulos[nulos > 0].to_dict() or "Nenhum"}')
print()
print('=== PRIMEIRAS LINHAS ===')
df[['CITY','STATE','REGION','IDHM','FAIXA_IDHM','GDP_CAPITA','IBGE_POP','PORTE']].head(4)

In [ ]:
# ── Estatísticas Descritivas Gerais ──────────────────────────────
print('=== ESTATÍSTICAS DESCRITIVAS ===')
df[['IDHM','IDHM_Renda','IDHM_Longevidade','IDHM_Educacao','GDP_CAPITA','IBGE_POP','DENSIDADE']].describe().round(3)

---
## 📊 2. Análise Univariada — Entendendo a Variável-Alvo

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Histograma ---
ax = axes[0]
n_bins = 40
counts, edges, patches = ax.hist(df['IDHM'], bins=n_bins, edgecolor='white', alpha=0.9)

# Colorir barras por faixa de IDHM
faixas_cores = list(zip(BINS_IDHM[:-1], BINS_IDHM[1:], CORES_FAIXA))
for patch, left in zip(patches, edges[:-1]):
    for fi, ff, cor in faixas_cores:
        if fi <= left < ff:
            patch.set_facecolor(cor)
            break

media   = df['IDHM'].mean()
mediana = df['IDHM'].median()
ax.axvline(media,   color='black', linestyle='--', lw=2, label=f'Média: {media:.3f}')
ax.axvline(mediana, color='gray',  linestyle=':',  lw=2, label=f'Mediana: {mediana:.3f}')
ax.set_title('Distribuição do IDHM', fontsize=11, fontweight='bold')
ax.set_xlabel('IDHM'); ax.set_ylabel('Municípios')
ax.legend(fontsize=9)
# Legenda de faixas
handles_f = [mpatches.Patch(color=c, label=l) for c, l in zip(CORES_FAIXA, LABELS_IDHM)]
ax.legend(handles=handles_f + ax.get_lines(), fontsize=7, loc='upper left')

# --- Boxplot ---
ax2 = axes[1]
ax2.boxplot(df['IDHM'].dropna(), patch_artist=True,
            boxprops=dict(facecolor='steelblue', alpha=0.6),
            medianprops=dict(color='red', linewidth=2),
            flierprops=dict(marker='o', markersize=2, alpha=0.3, color='gray'))
q1, q3 = df['IDHM'].quantile([0.25, 0.75])
ax2.annotate(f'Q3: {q3:.3f}', xy=(1, q3), xytext=(1.25, q3), fontsize=9)
ax2.annotate(f'Q1: {q1:.3f}', xy=(1, q1), xytext=(1.25, q1), fontsize=9)
ax2.annotate(f'Mediana: {mediana:.3f}', xy=(1, mediana), xytext=(1.25, mediana+0.005), fontsize=9, color='red')
ax2.set_title('Boxplot do IDHM', fontsize=11, fontweight='bold')
ax2.set_ylabel('IDHM'); ax2.set_xticks([])

# --- Pizza por faixa ---
ax3 = axes[2]
contagem_faixa = df['FAIXA_IDHM'].value_counts().sort_index()
wedges, texts, autotexts = ax3.pie(
    contagem_faixa.values,
    labels=contagem_faixa.index,
    colors=CORES_FAIXA,
    autopct=lambda p: f'{p:.1f}%\n({int(p/100*len(df)):,})',
    startangle=90, pctdistance=0.75,
    wedgeprops=dict(edgecolor='white', linewidth=1.5)
)
for t in autotexts: t.set_fontsize(8)
ax3.set_title('Municípios por Faixa de IDHM', fontsize=11, fontweight='bold')

plt.suptitle('Análise Univariada do IDHM — Municípios Brasileiros',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../projeto-final/figuras/pf_fig1_univariada.png', bbox_inches='tight')
plt.show()

print(f'\n📊 RESUMO ESTATÍSTICO:')
print(f'  Média        : {media:.3f}  |  Mediana    : {mediana:.3f}')
print(f'  Desvio-padrão: {df["IDHM"].std():.3f}  |  IQR        : {(q3-q1):.3f}')
print(f'  Min          : {df["IDHM"].min():.3f}  |  Max        : {df["IDHM"].max():.3f}')
print(f'\n  Assimetria (skewness): {df["IDHM"].skew():.3f}')
print('  (Valor positivo indica cauda à direita; mais municípios abaixo da média)')

---
## 🗺️ 3. Pergunta 1 — Distribuição Geográfica do IDHM

In [ ]:
# Estatísticas por Região
stats_reg = (
    df.groupby('REGION')['IDHM']
    .agg(Média='mean', Mediana='median', DP='std', Mín='min', Máx='max', N='count')
    .round(3)
    .sort_values('Mediana', ascending=False)
)
print('=== IDHM POR REGIÃO ===')
display(stats_reg)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# --- Violinplot por Região ---
ax = axes[0]
ordem_reg = stats_reg.index.tolist()
dados_vio  = [df[df['REGION']==r]['IDHM'].dropna().values for r in ordem_reg]
partes = ax.violinplot(dados_vio, showmedians=True, showextrema=True)
for pc, reg in zip(partes['bodies'], ordem_reg):
    pc.set_facecolor(COR_REGIAO[reg]); pc.set_alpha(0.75)
partes['cmedians'].set_color('black'); partes['cmedians'].set_linewidth(2)
ax.set_xticks(range(1, len(ordem_reg)+1))
ax.set_xticklabels(ordem_reg, fontsize=9)
ax.set_title('Distribuição do IDHM por Região\n(violinplot mostra forma + mediana)',
             fontsize=11, fontweight='bold')
ax.set_ylabel('IDHM')

# --- IDHM médio por Estado (todos os estados) ---
ax2 = axes[1]
idhm_est = (
    df.groupby(['STATE', 'REGION'])['IDHM']
    .mean().reset_index().sort_values('IDHM')
)
cores_est = [COR_REGIAO[r] for r in idhm_est['REGION']]
barras = ax2.barh(idhm_est['STATE'], idhm_est['IDHM'], color=cores_est, edgecolor='white', height=0.7)
ax2.axvline(df['IDHM'].mean(), color='black', linestyle='--', linewidth=1.5, label='Média Nacional')
for b, v in zip(barras, idhm_est['IDHM']):
    ax2.text(v + 0.002, b.get_y()+b.get_height()/2, f'{v:.3f}', va='center', fontsize=7)
ax2.set_title('IDHM Médio por Estado', fontsize=11, fontweight='bold')
ax2.set_xlabel('IDHM Médio')
ax2.set_xlim(0.53, 0.83)
handles_r = [mpatches.Patch(color=c, label=r) for r, c in COR_REGIAO.items()]
ax2.legend(handles=handles_r, title='Região', fontsize=8, loc='lower right')

plt.suptitle('🗺️ Pergunta 1: Como o IDHM varia geograficamente?',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../projeto-final/figuras/pf_fig2_geografica.png', bbox_inches='tight')
plt.show()

print('\n💡 INSIGHT 1:')
print('  A região Sul apresenta a maior mediana de IDHM, seguida pelo Sudeste.')
print('  O Nordeste tem não apenas a menor mediana, mas também a maior dispersão,')
print('  indicando forte heterogeneidade interna — coexistem municípios relativamente')
print('  desenvolvidos (capitais) com municípios de IDHM muito baixo no interior.')

---
## 🔗 4. Pergunta 2 — Quais Variáveis se Associam ao IDHM?

In [ ]:
cols_corr = ['IDHM','IDHM_Renda','IDHM_Longevidade','IDHM_Educacao',
             'GDP_CAPITA','LOG_GDP_CAPITA','IBGE_POP','DENSIDADE']
cols_corr = [c for c in cols_corr if c in df.columns]

corr = df[cols_corr].corr(method='pearson').round(3)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# --- Heatmap ---
ax = axes[0]
mascara = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            mask=mascara, ax=ax, linewidths=0.5, vmin=-1, vmax=1,
            square=True, cbar_kws={'label': 'r de Pearson', 'shrink': 0.8})
ax.set_title('Matriz de Correlação de Pearson', fontsize=11, fontweight='bold')

# --- Gráfico de correlação com IDHM (barras) ---
ax2 = axes[1]
excluir_da_corr = {'IDHM', 'IDHM_Renda', 'IDHM_Longevidade', 'IDHM_Educacao'}
corr_idhm = corr['IDHM'].drop(labels=list(excluir_da_corr & set(corr.index)))
corr_idhm = corr_idhm.sort_values()

cores_corr = ['#d62728' if v < 0 else '#2ca02c' for v in corr_idhm.values]
barras = ax2.barh(corr_idhm.index, corr_idhm.values, color=cores_corr, edgecolor='white')
ax2.axvline(0, color='black', linewidth=1)
for b, v in zip(barras, corr_idhm.values):
    offset = 0.01 if v >= 0 else -0.01
    ha = 'left' if v >= 0 else 'right'
    ax2.text(v + offset, b.get_y()+b.get_height()/2, f'{v:.3f}', va='center', ha=ha, fontsize=10)

ax2.set_title('Correlação das Variáveis com IDHM\n(excluindo subcomponentes do IDHM)',
              fontsize=11, fontweight='bold')
ax2.set_xlabel('Correlação de Pearson com IDHM')

plt.suptitle('🔗 Pergunta 2: Quais variáveis se correlacionam com o IDHM?',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../projeto-final/figuras/pf_fig3_correlacao.png', bbox_inches='tight')
plt.show()

mais_correlacionada = corr_idhm.abs().idxmax()
print(f'\n💡 INSIGHT 2:')
print(f'  A variável não-componente com maior correlação com IDHM é: {mais_correlacionada}')
print(f'  (r = {corr_idhm[mais_correlacionada]:.3f})')
print('  Isso reforça que a prosperidade econômica e o desenvolvimento humano')
print('  estão fortemente ligados — mas a relação não é perfeita (r < 1).')

In [ ]:
# Scatter PIB per Capita x IDHM — análise detalhada
fig, ax = plt.subplots(figsize=(12, 7))

p95 = df['GDP_CAPITA'].quantile(0.95)
df_sc = df[df['GDP_CAPITA'] <= p95].dropna(subset=['GDP_CAPITA','IDHM'])

# Plotando por região
for reg, grp in df_sc.groupby('REGION'):
    ax.scatter(grp['GDP_CAPITA'], grp['IDHM'], color=COR_REGIAO[reg],
               label=reg, alpha=0.35, s=16, edgecolors='none')

# Linha de tendência
coefs = np.polyfit(df_sc['GDP_CAPITA'], df_sc['IDHM'], 1)
x_r = np.linspace(df_sc['GDP_CAPITA'].min(), df_sc['GDP_CAPITA'].max(), 300)
ax.plot(x_r, np.polyval(coefs, x_r), 'k--', lw=2, zorder=5, label='Tendência linear')

# Quadrantes
pib_med  = df_sc['GDP_CAPITA'].median()
idhm_med = df_sc['IDHM'].median()
ax.axvline(pib_med,  color='gray', lw=0.8, linestyle=':')
ax.axhline(idhm_med, color='gray', lw=0.8, linestyle=':')
kw = dict(fontsize=8, color='gray', style='italic', alpha=0.8)
ax.text(pib_med*0.05,  df_sc['IDHM'].max()*0.99, 'Baixo PIB /\nAlto IDHM', **kw)
ax.text(pib_med*1.05,  df_sc['IDHM'].max()*0.99, 'Alto PIB /\nAlto IDHM', **kw)
ax.text(pib_med*0.05,  df_sc['IDHM'].min()*1.01, 'Baixo PIB /\nBaixo IDHM', **kw)
ax.text(pib_med*1.05,  df_sc['IDHM'].min()*1.01, 'Alto PIB /\nBaixo IDHM', **kw)

ax.set_xlabel('PIB per Capita (R$)', fontsize=11)
ax.set_ylabel('IDHM', fontsize=11)
ax.set_title('PIB per Capita × IDHM — Municípios Brasileiros por Região',
             fontsize=12, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'R$ {x/1000:.0f}k'))
ax.legend(title='Região', framealpha=0.85)

plt.tight_layout()
plt.savefig('../projeto-final/figuras/pf_fig4_scatter_pib_idhm.png', bbox_inches='tight')
plt.show()

---
## ⚖️ 5. Pergunta 3 — Desigualdade Interna nos Estados

In [ ]:
# Desvio-padrão do IDHM por estado = indicador de desigualdade interna
desig = (
    df.groupby(['STATE','REGION'])['IDHM']
    .agg(Media='mean', DP='std', Min='min', Max='max', N='count')
    .reset_index()
    .sort_values('DP', ascending=False)
    .round(3)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Barras: desvio-padrão por estado ---
ax = axes[0]
cores_d = [COR_REGIAO[r] for r in desig['REGION']]
ax.bar(desig['STATE'], desig['DP'], color=cores_d, edgecolor='white', alpha=0.85)
ax.set_title('Desigualdade Interna do IDHM por Estado\n(Desvio-Padrão entre municípios)',
             fontsize=11, fontweight='bold')
ax.set_xlabel('Estado (UF)'); ax.set_ylabel('Desvio-Padrão do IDHM')
ax.tick_params(axis='x', rotation=70, labelsize=8)
handles_r = [mpatches.Patch(color=c, label=r) for r, c in COR_REGIAO.items()]
ax.legend(handles=handles_r, title='Região', fontsize=8)

# --- Range IDHM por Estado (min-max range plot) ---
ax2 = axes[1]
desig_sorted = desig.sort_values('Media', ascending=True)
y_pos = range(len(desig_sorted))
cores_est2 = [COR_REGIAO[r] for r in desig_sorted['REGION']]
ax2.hlines(y_pos, desig_sorted['Min'], desig_sorted['Max'], color=cores_est2, linewidth=3, alpha=0.6)
ax2.scatter(desig_sorted['Media'], y_pos, color=cores_est2, s=50, zorder=5)
ax2.set_yticks(y_pos)
ax2.set_yticklabels(desig_sorted['STATE'], fontsize=8)
ax2.set_title('Range do IDHM por Estado\n(ponto = média, linha = min–max)',
              fontsize=11, fontweight='bold')
ax2.set_xlabel('IDHM')
ax2.legend(handles=handles_r, title='Região', fontsize=8, loc='lower right')

plt.suptitle('⚖️ Pergunta 3: Quão desiguais são os estados internamente?',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../projeto-final/figuras/pf_fig5_desigualdade.png', bbox_inches='tight')
plt.show()

top3 = desig.head(3)
print('\n💡 INSIGHT 3 — Top 3 estados com maior desigualdade interna:')
for _, r in top3.iterrows():
    print(f'  {r["STATE"]} ({r["REGION"]}): DP={r["DP"]:.3f}, range=[{r["Min"]:.3f}–{r["Max"]:.3f}]')
print('\n  Nesses estados convivem municípios com IDHM de países diferentes.')
print('  A média mascara uma realidade de extrema heterogeneidade territorial.')

---
## 🏙️ 6. Perguntas 4 e 6 — Urbanização, Porte e Desenvolvimento

In [ ]:
# IDHM por porte e contagem de municípios
idhm_porte = (
    df.groupby('PORTE', observed=True)['IDHM']
    .agg(IDHM_Medio='mean', Desvio='std', N='count')
    .reset_index().round(3)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- IDHM médio por porte ---
ax = axes[0]
palette = sns.color_palette('Blues_d', n_colors=len(idhm_porte))
bs = ax.bar(idhm_porte['PORTE'], idhm_porte['IDHM_Medio'],
            color=palette, edgecolor='white',
            yerr=idhm_porte['Desvio'], capsize=5,
            error_kw={'ecolor': '#555', 'lw': 1.5})
ax.set_title('IDHM Médio por Porte do Município\n(barra de erro = ±1 desvio-padrão)',
             fontsize=11, fontweight='bold')
ax.set_ylabel('IDHM Médio'); ax.set_ylim(0.5, 0.92)
for b, row in zip(bs, idhm_porte.itertuples()):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.022,
            f'{row.IDHM_Medio:.3f}', ha='center', fontsize=9, fontweight='bold')

# --- Número de municípios por porte ---
ax2 = axes[1]
bs2 = ax2.bar(idhm_porte['PORTE'], idhm_porte['N'], color=palette, edgecolor='white')
for b, row in zip(bs2, idhm_porte.itertuples()):
    pct = row.N / len(df) * 100
    ax2.text(b.get_x()+b.get_width()/2, b.get_height()+15,
             f'{row.N:,}\n({pct:.1f}%)', ha='center', fontsize=8)
ax2.set_title('Quantidade de Municípios por Porte', fontsize=11, fontweight='bold')
ax2.set_ylabel('Número de Municípios')

plt.suptitle('🏙️ Perguntas 4 e 6: Tamanho Importa para o IDHM?',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../projeto-final/figuras/pf_fig6_porte.png', bbox_inches='tight')
plt.show()

print('\n💡 INSIGHT 4 — Porte e Desenvolvimento:')
print('  A grande maioria dos municípios brasileiros é de pequeno porte (<20k hab),')
print('  mas metrópoles têm IDHM médio significativamente superior.')
print('  Porém, a alta variação interna (barras de erro grandes) revela que')
print('  tamanho é uma tendência, não uma garantia — existem exceções relevantes.')

---
## 🧩 7. Pergunta 5 — Componentes do IDHM: Onde Está o Gargalo?

In [ ]:
comp_cols = ['IDHM_Renda', 'IDHM_Longevidade', 'IDHM_Educacao']
comp_cols = [c for c in comp_cols if c in df.columns]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# --- Barras agrupadas: média por região ---
ax = axes[0]
comp_reg = (
    df.groupby('REGION')[comp_cols].mean()
    .round(3)
    .loc[['Norte','Nordeste','Centro-Oeste','Sudeste','Sul']]
)
comp_reg.columns = ['Renda', 'Longevidade', 'Educação']
x      = np.arange(len(comp_reg))
width  = 0.25
cores_comp = ['#e6994c', '#56b4e9', '#009e73']
for i, (col, cor) in enumerate(zip(comp_reg.columns, cores_comp)):
    bars = ax.bar(x + i*width - width, comp_reg[col], width, label=col, color=cor, edgecolor='white')
    for b in bars:
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.003,
                f'{b.get_height():.3f}', ha='center', fontsize=7, rotation=90)
ax.set_xticks(x); ax.set_xticklabels(comp_reg.index, fontsize=9)
ax.set_title('Componentes do IDHM por Região', fontsize=11, fontweight='bold')
ax.set_ylabel('Valor do Componente'); ax.set_ylim(0.45, 0.88)
ax.legend(title='Componente')
ax.axhline(df['IDHM'].mean(), color='black', linestyle='--', lw=1, alpha=0.5, label='IDHM médio')

# --- Radar chart (spider) nacional ---
ax2 = axes[1]
medias_comp = df[comp_cols].mean()
medias_comp.index = ['Renda', 'Longevidade', 'Educação']
medias_comp['IDHM Total'] = df['IDHM'].mean()

# Gráfico de barras horizontais simples como alternativa ao radar
cores_b = ['#e6994c', '#56b4e9', '#009e73', '#8c564b']
hs = ax2.barh(medias_comp.index, medias_comp.values, color=cores_b, edgecolor='white', height=0.5)
for h, v in zip(hs, medias_comp.values):
    ax2.text(v + 0.005, h.get_y()+h.get_height()/2, f'{v:.3f}', va='center', fontsize=11, fontweight='bold')
ax2.set_xlim(0.5, 0.83)
ax2.set_title('Média Nacional dos Componentes\ndo IDHM (todos os municípios)',
              fontsize=11, fontweight='bold')
ax2.set_xlabel('Valor Médio')

plt.suptitle('🧩 Pergunta 5: Qual é o Gargalo do IDHM no Brasil?',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../projeto-final/figuras/pf_fig7_componentes.png', bbox_inches='tight')
plt.show()

gargalo = medias_comp.drop('IDHM Total').idxmin()
print(f'\n💡 INSIGHT 5:')
print(f'  O gargalo nacional do IDHM é o componente: "{gargalo}"')
print(f'  (menor média entre os três componentes em nível nacional)')
print('  Enquanto Longevidade avançou com políticas de saúde pública,')
print('  Educação ainda apresenta os índices mais críticos — especialmente no Norte e Nordeste.')

---
## 🌐 8. Análise Multivariada — Visão Integradora

In [ ]:
# Pairplot dos principais indicadores por região
# Mostra todas as relações bivariadas de uma vez
cols_pair = ['IDHM','IDHM_Educacao','IDHM_Renda','GDP_CAPITA']
cols_pair = [c for c in cols_pair if c in df.columns]

df_pair = df[cols_pair + ['REGION']].dropna()
if len(df_pair) > 1000:
    df_pair = df_pair.sample(1000, random_state=42)

renomear = {'IDHM': 'IDHM Total', 'IDHM_Educacao': 'IDHM Educ.',
            'IDHM_Renda': 'IDHM Renda', 'GDP_CAPITA': 'PIB per Cap.'}
df_pair = df_pair.rename(columns=renomear)

pg = sns.pairplot(
    df_pair, hue='REGION',
    palette=COR_REGIAO,
    plot_kws={'alpha': 0.35, 's': 15},
    diag_kind='kde',
    corner=True
)
pg.figure.suptitle('🌐 Pairplot: Relações entre Indicadores por Região',
                   fontsize=13, fontweight='bold', y=1.01)
pg.figure.set_size_inches(11, 9)
plt.savefig('../projeto-final/figuras/pf_fig8_pairplot.png', bbox_inches='tight')
plt.show()
print('📊 Pairplot gerado. Observe:')
print('  - Diagonal: distribuição de cada variável por região (KDE)')
print('  - Triângulo inferior: scatter plot entre pares de variáveis')
print('  - As nuvens de pontos por cor mostram como cada região se posiciona')

---
## 📊 9. Painel Executivo — Dashboard Resumo

In [ ]:
fig = plt.figure(figsize=(18, 11))
fig.patch.set_facecolor('#f8f9fa')

# Título e subtítulo
fig.text(0.5, 0.97, 'Desenvolvimento Humano nos Municípios Brasileiros',
         ha='center', fontsize=17, fontweight='bold', color='#1a1a2e')
fig.text(0.5, 0.945, 'Uma análise exploratória do IDHM e seus determinantes | Brazilian Cities Dataset',
         ha='center', fontsize=10, color='gray', style='italic')

gs = fig.add_gridspec(3, 4, hspace=0.55, wspace=0.38,
                      top=0.92, bottom=0.06, left=0.06, right=0.97)

# ── KPIs numéricos (linha 0, 4 cards) ─────────────────────────────
kpis = [
    ('IDHM\nMédio Nacional', f'{df["IDHM"].mean():.3f}', '#3498db'),
    ('Municípios\nAnalisados', f'{len(df):,}', '#2ecc71'),
    ('IDHM Médio\nNordeste', f'{df[df["REGION"]=="Nordeste"]["IDHM"].mean():.3f}', '#e67e22'),
    ('IDHM Médio\nSul', f'{df[df["REGION"]=="Sul"]["IDHM"].mean():.3f}', '#9b59b6'),
]
for col_idx, (label, valor, cor) in enumerate(kpis):
    ax_kpi = fig.add_subplot(gs[0, col_idx])
    ax_kpi.set_facecolor(cor)
    ax_kpi.text(0.5, 0.62, valor, ha='center', va='center', fontsize=22,
                fontweight='bold', color='white', transform=ax_kpi.transAxes)
    ax_kpi.text(0.5, 0.22, label, ha='center', va='center', fontsize=9,
                color='white', transform=ax_kpi.transAxes)
    ax_kpi.set_xticks([]); ax_kpi.set_yticks([])
    for spine in ax_kpi.spines.values(): spine.set_visible(False)

# ── Gráfico 1: Histograma (linha 1, colunas 0-1) ───────────────────
ax1 = fig.add_subplot(gs[1, 0:2])
n_hist, bins_hist, patches_hist = ax1.hist(df['IDHM'], bins=35, edgecolor='white', alpha=0.9)
for patch, left in zip(patches_hist, bins_hist[:-1]):
    for fi, ff, cor in zip(BINS_IDHM[:-1], BINS_IDHM[1:], CORES_FAIXA):
        if fi <= left < ff: patch.set_facecolor(cor); break
ax1.axvline(df['IDHM'].mean(), color='black', lw=2, linestyle='--',
            label=f'Média: {df["IDHM"].mean():.3f}')
ax1.set_title('① Distribuição do IDHM por Faixa', fontsize=10, fontweight='bold')
ax1.set_xlabel('IDHM'); ax1.set_ylabel('Municípios')
ax1.legend(fontsize=8)

# ── Gráfico 2: IDHM por Região (linha 1, colunas 2-3) ─────────────
ax2 = fig.add_subplot(gs[1, 2:4])
med_reg2 = df.groupby('REGION')['IDHM'].mean().sort_values()
cores_r2 = [COR_REGIAO[r] for r in med_reg2.index]
barras2 = ax2.barh(med_reg2.index, med_reg2.values, color=cores_r2, edgecolor='white', height=0.6)
ax2.axvline(df['IDHM'].mean(), color='black', lw=1.5, linestyle='--', alpha=0.6)
for b, v in zip(barras2, med_reg2.values):
    ax2.text(v + 0.002, b.get_y()+b.get_height()/2, f'{v:.3f}', va='center', fontsize=9)
ax2.set_title('② IDHM Médio por Região', fontsize=10, fontweight='bold')
ax2.set_xlabel('IDHM Médio'); ax2.set_xlim(0.55, 0.82)

# ── Gráfico 3: Componentes (linha 2, colunas 0-1) ─────────────────
ax3 = fig.add_subplot(gs[2, 0:2])
if comp_cols:
    cr = df.groupby('REGION')[comp_cols].mean().round(3)
    cr.columns = ['Renda','Longevidade','Educação']
    cr = cr.loc[['Norte','Nordeste','Centro-Oeste','Sudeste','Sul']]
    cr.plot(kind='bar', ax=ax3, width=0.7, color=['#e6994c','#56b4e9','#009e73'], edgecolor='white')
ax3.set_title('③ Componentes do IDHM por Região', fontsize=10, fontweight='bold')
ax3.set_ylabel('Valor'); ax3.tick_params(axis='x', rotation=20, labelsize=8)
ax3.set_ylim(0.45, 0.87)
ax3.legend(title='Componente', fontsize=7, loc='upper left')

# ── Gráfico 4: Scatter (linha 2, colunas 2-3) ─────────────────────
ax4 = fig.add_subplot(gs[2, 2:4])
df_sc4 = df[df['GDP_CAPITA'] <= df['GDP_CAPITA'].quantile(0.95)].dropna(subset=['GDP_CAPITA','IDHM'])
if len(df_sc4) > 600: df_sc4 = df_sc4.sample(600, random_state=1)
for reg, grp in df_sc4.groupby('REGION'):
    ax4.scatter(grp['GDP_CAPITA'], grp['IDHM'], color=COR_REGIAO[reg], alpha=0.3, s=10)
coefs4 = np.polyfit(df_sc4['GDP_CAPITA'], df_sc4['IDHM'], 1)
x4 = np.linspace(df_sc4['GDP_CAPITA'].min(), df_sc4['GDP_CAPITA'].max(), 200)
ax4.plot(x4, np.polyval(coefs4, x4), 'k--', lw=1.8)
ax4.set_title('④ PIB per Capita × IDHM', fontsize=10, fontweight='bold')
ax4.set_xlabel('PIB per Capita (R$)'); ax4.set_ylabel('IDHM')
ax4.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'R${x/1000:.0f}k'))
handles_r4 = [mpatches.Patch(color=c, label=r) for r, c in COR_REGIAO.items()]
ax4.legend(handles=handles_r4, fontsize=7, ncol=2)

plt.savefig('../projeto-final/figuras/pf_fig9_dashboard.png', bbox_inches='tight', dpi=130)
plt.show()
print('✅ Dashboard executivo gerado!')

---
## 📖 10. Conclusões e Insights

### 🔑 Síntese dos Principais Achados

---

#### Insight 1 — O Brasil tem dois ritmos de desenvolvimento
A análise geográfica revelou uma divisão persistente: **Sul e Sudeste** concentram municípios com IDHM acima da média nacional (>0,73), enquanto **Norte e Nordeste** ficam consistentemente abaixo (0,60–0,62). Essa não é apenas uma estatística — representa diferenças reais em expectativa de vida, acesso à educação e oportunidades econômicas para milhões de brasileiros.

---

#### Insight 2 — PIB per capita é o principal preditor do IDHM
Entre as variáveis analisadas, o **PIB per capita** apresentou a maior correlação linear com o IDHM. Entretanto, a dispersão elevada no scatter plot evidencia que a riqueza econômica não garante automaticamente bem-estar coletivo: existem municípios com PIB elevado e IDHM moderado, provavelmente por conta da **concentração de renda** que não se traduz em serviços públicos e qualidade de vida para todos.

---

#### Insight 3 — Desigualdade dentro dos estados é tão importante quanto entre regiões
A análise por estado revelou que o desvio-padrão interno do IDHM é alto em vários estados — especialmente aqueles com capitais desenvolvidas rodeadas por municípios de interior com IDHM muito baixo. **Um estado pode ter média razoável e ainda assim abrigar municípios em situação crítica**. Políticas públicas baseadas apenas em médias estaduais podem mascarar essa realidade.

---

#### Insight 4 — Educação é o gargalo nacional
O componente **Educação** é o mais baixo do IDHM em todas as regiões. Enquanto a **Longevidade** avançou significativamente graças ao SUS e à vacinação, a educação ainda apresenta déficits estruturais. Isso indica que investimentos em educação têm o maior potencial de impacto sobre o desenvolvimento humano nos próximos anos.

---

#### Insight 5 — Municípios menores são a maioria, mas os mais vulneráveis
Mais de 70% dos municípios brasileiros têm menos de 20 mil habitantes. Esses municípios têm, em média, IDHM mais baixo e menor capacidade de arrecadação e investimento. Políticas públicas de escala nacional precisam considerar esse perfil predominante — e não apenas grandes cidades — para serem efetivas.

---

### 🏛️ Recomendações de Política Pública

Com base nos dados analisados, três recomendações emergem com maior clareza:

1. **Prioridade em Educação no Norte e Nordeste**: o gargalo da Educação é mais severo nessas regiões. Programas de infraestrutura escolar, formação de professores e combate à evasão escolar devem ser intensificados nos municípios de menor IDHM educacional.

2. **Abordagem territorial e não apenas estadual**: como a desigualdade interna é alta, políticas devem identificar municípios em situação crítica dentro de estados de média razoável, evitando que recursos sejam alocados apenas nas capitais.

3. **Políticas de renda redistributiva**: a correlação entre PIB per capita e IDHM é forte, mas imperfeita. Para que a riqueza econômica se converta em desenvolvimento humano, são necessárias políticas que aumentem a participação da população mais pobre nos ganhos econômicos locais.

---

### 🔭 Próximos Passos

Esta EDA abre várias possibilidades de análise mais aprofundada:

- **Análise temporal**: como o IDHM evoluiu entre os Censos de 1991, 2000 e 2010?
- **Modelagem preditiva**: quais variáveis explicam *causalmente* as variações no IDHM?
- **Clustering**: quais grupos de municípios compartilham perfis socioeconômicos similares?
- **Análise espacial**: municípios vizinhos têm IDHM parecido? (autocorrelação espacial — Índice de Moran)

---

### 💬 Reflexão Final

> *"Dados não falam por si só. Eles precisam de boas perguntas, análises honestas e comunicação clara para se tornarem insights úteis. Esta análise foi uma tentativa de olhar para números e enxergar pessoas — municípios com histórias de desigualdade, esforço e possibilidade de transformação."*

---

**Equipe:** *(insira os nomes dos integrantes aqui)*  
**Curso:** Ciência de Dados — T326 | UNIFOR  
**Ano:** 2025

In [ ]:
# Resumo final — tabela síntese dos insights
print('=' * 65)
print('  RESUMO EXECUTIVO — PRINCIPAIS ACHADOS DA EDA')
print('=' * 65)

print(f'\n  📍 Municípios analisados      : {len(df):,}')
print(f'  📍 IDHM médio nacional         : {df["IDHM"].mean():.3f}')
print(f'  📍 IDHM mediana nacional       : {df["IDHM"].median():.3f}')
print(f'  📍 Região com maior IDHM médio : {df.groupby("REGION")["IDHM"].mean().idxmax()}')
print(f'  📍 Região com menor IDHM médio : {df.groupby("REGION")["IDHM"].mean().idxmin()}')

if comp_cols:
    garg = df[comp_cols].mean().idxmin().replace('IDHM_', '')
    print(f'  📍 Componente gargalo (nacional): {garg}')

corr_idhm_ext = df[['IDHM','GDP_CAPITA','IBGE_POP','DENSIDADE']].corr()['IDHM'].drop('IDHM')
print(f'  📍 Var. mais correlac. ao IDHM : {corr_idhm_ext.abs().idxmax()}')
print(f'     (r = {corr_idhm_ext[corr_idhm_ext.abs().idxmax()]:.3f})')

pct_pequenos = (df['IBGE_POP'] < 20000).mean() * 100
print(f'  📍 Municípios com <20k hab.    : {pct_pequenos:.1f}%')

print()
print('  → Consulte os gráficos e conclusões acima para a narrativa completa.')
print('=' * 65)